# OTTO Recommender Systems EDA

## 결론 요약

| 항목 | 내용 |
|---|---|
| 데이터셋 | OTTO Recommender Systems Dataset (Kaggle) |
| 이벤트 유형 | clicks / carts / orders |
| 총 세션 수 | ~12.9M (전체) / 분석 시 100만 샘플 사용 예정 |
| 용도 | 세션 기반 의도 예측 — clicks→orders 시퀀스 |

### Retailrocket 대비 차이점
| 항목 | Retailrocket | OTTO |
|---|---|---|
| 도메인 | 일반 이커머스 | 일반 이커머스 |
| 이벤트 구조 | view/addtocart/transaction | clicks/carts/orders |
| 세션 ID | 직접 계산 (30분 gap) | 제공됨 |
| 데이터 크기 | 280만 이벤트 | 2.2억 이벤트 |
| 호텔 도메인 적합성 | 낮음 (동일) | 낮음 |

### 본 프로젝트 활용 결론
- OTTO는 세션 기반 추천 연구의 표준 데이터셋으로, clicks→orders 전환 패턴 분석에 적합
- **`session_length_5min`, `hidden_repeated` booster 가중치 검증**: Retailrocket 결과(lift 32x, 8.76x)와 동일 방향성 확인 예정
- 호텔 예약 특유 행동(날짜 변경, 객실 옵션 비교)은 두 데이터셋 모두에 없음 → 도메인 직관 유지
- **데이터 미확보로 인해 실행 보류** — 12GB 다운로드 후 §1~3 셀 실행 가능

In [ ]:
# OTTO 데이터 다운로드 방법 (터미널에서 실행)
# cd notebooks/data
# kaggle competitions download -c otto-recommender-system
# tar -xf otto-recommender-system.zip

import os
OTTO_PATH = 'data/otto/train.parquet'
DATA_AVAILABLE = os.path.exists(OTTO_PATH)
print(f'OTTO 데이터 사용 가능: {DATA_AVAILABLE}')
if not DATA_AVAILABLE:
    print('⚠ data/otto/ 폴더에 train.parquet 없음 — 다운로드 필요')

## §1. 데이터 로딩 및 기초 통계
*(데이터 다운로드 후 실행)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6), 'figure.dpi': 150})

if not DATA_AVAILABLE:
    print('데이터 없음 — 스킵'); raise SystemExit

train = pd.read_parquet(OTTO_PATH)
# 너무 크면 샘플링
if len(train) > 1_000_000:
    train = train.sample(n=1_000_000, random_state=42)
    print(f'샘플링 완료: {len(train):,}행')

print(f'컬럼: {train.columns.tolist()}')
print(f'이벤트 유형:\n{train["type"].value_counts()}')
print(f'고유 세션 수: {train["session"].nunique():,}')

## §2. 세션 통계 및 전환율

In [ ]:
if not DATA_AVAILABLE: raise SystemExit

session_stats = train.groupby('session').agg(
    event_count=('aid', 'count'),
    has_cart=('type', lambda x: 'cart' in x.values),
    has_order=('type', lambda x: 'order' in x.values)
).reset_index()

print(session_stats.describe())
print(f"\n카트 있는 세션: {session_stats['has_cart'].mean()*100:.2f}%")
print(f"주문 있는 세션: {session_stats['has_order'].mean()*100:.2f}%")
print(f"카트→주문 전환율: {(session_stats['has_cart'] & session_stats['has_order']).sum() / session_stats['has_cart'].sum()*100:.2f}%")

## §3. Retailrocket 결과와의 비교 검증

In [ ]:
if not DATA_AVAILABLE: raise SystemExit

# 세션 길이별 주문율 (session_length_5min booster 검증)
# OTTO는 timestamp 컬럼이 있으면 duration 계산 가능
if 'ts' in train.columns:
    duration = train.groupby('session')['ts'].agg(lambda x: (x.max()-x.min())/1000)
    session_stats['duration_sec'] = session_stats['session'].map(duration)
    session_stats['is_long'] = session_stats['duration_sec'] >= 300

    long = session_stats[session_stats['is_long']]['has_order'].mean()
    short = session_stats[~session_stats['is_long']]['has_order'].mean()
    lift = long / short if short > 0 else None
    print(f'5분+ 세션 주문율: {long*100:.2f}%')
    print(f'5분- 세션 주문율: {short*100:.2f}%')
    print(f'Lift: {lift:.2f}x (Retailrocket: 32.19x 참고)')
else:
    print('ts 컬럼 없음 — duration 계산 불가')

## §4. 결론 및 thresholds.yml 영향

### 호텔 도메인 매핑 한계
- OTTO는 상품 ID(`aid`)만 있고 카테고리·가격 정보 없음
- 호텔 예약 특유 행동(날짜 변경, 객실 비교) 패턴 없음
- 클립보드 복사·탭 전환 신호 없음

### thresholds.yml 반영 여부
| 항목 | OTTO 분석 결과 | 반영 |
|---|---|---|
| session_length_5min | 실행 후 Retailrocket과 비교 | W3 값(0.4) 유지 또는 조정 |
| hidden_repeated | 실행 후 확인 | W3 값(0.4) 유지 또는 조정 |
| intent_score_min | OTTO로 검증 불가 | 0.6 유지 (W5 A/B 결과로 결정) |

**데이터 확보 시 §1~3 셀 실행 후 이 셀의 표를 업데이트할 것.**